# LegacyLens – train the document classifier on Kaggle (free GPU)
1. Settings → Accelerator → **GPU T4 x2**, Internet → **On**
2. Upload the repo zip as a Kaggle *Dataset* (Add Data → Upload) or `git clone` your GitHub repo
3. Run all cells, then download `models/doc_classifier.onnx` + `models/doc_classifier.json` from the Output tab

In [ ]:
!git clone https://github.com/<your-user>/legacylens-ai.git || true
%cd legacylens-ai
!pip install -q presidio-analyzer pillow scikit-image onnx datasets

In [ ]:
# 1) synthetic data: 6 classes x 600 images (~10 min)
!PYTHONPATH=src python -m legacylens_ai.data.synthetic_docs --out data/synthetic --n-per-class 600

In [ ]:
# 2) OPTIONAL but important: add REAL public images to fight the synthetic->real gap
# RVL-CDIP on Hugging Face (aharley/rvl_cdip): letters/memos/handwritten -> other, invoices/budgets -> financial
import os
from datasets import load_dataset
ds = load_dataset('aharley/rvl_cdip', split='train', streaming=True)
names = ds.features['label'].names
mapping = {'letter': 'other', 'memo': 'other', 'handwritten': 'other', 'invoice': 'financial', 'budget': 'financial'}
PER_CLASS = 400
count = {'other': 0, 'financial': 0}
for ex in ds:
    cls = mapping.get(names[ex['label']])
    if cls is None or count[cls] >= PER_CLASS:
        if all(v >= PER_CLASS for v in count.values()):
            break
        continue
    count[cls] += 1
    i = count[cls]
    split = 'train' if i <= 0.8 * PER_CLASS else ('val' if i <= 0.9 * PER_CLASS else 'test')
    d = f'data/synthetic/images/{split}/{cls}'
    os.makedirs(d, exist_ok=True)
    ex['image'].convert('RGB').save(f'{d}/rvl_{i:05d}.jpg')
print(count)

In [ ]:
# 3) train EfficientNet-B0 + export ONNX
!python scripts/train_classifier_torch.py --data data/synthetic --epochs 8

In [ ]:
!ls -la models/